# Machine Learning Fundamentals Assignment
### Topics: Weights, Activation Functions, Probability Distributions, Gradients & Logistic Regression with TensorFlow

---
## Q1: Role of Weights in a Neuron

### What are Weights?
Weights are learnable numerical parameters associated with each connection between neurons in a neural network. Every input to a neuron has a corresponding weight that determines **how much influence** that input has on the neuron's output.

### Mathematical Representation
For a single neuron receiving inputs $x_1, x_2, ..., x_n$ with weights $w_1, w_2, ..., w_n$ and bias $b$:

$$z = w_1x_1 + w_2x_2 + ... + w_nx_n + b = \sum_{i=1}^{n} w_i x_i + b$$

The output (after activation) is: $a = f(z)$

### Role of Weights

| Role | Description |
|------|-------------|
| **Signal Amplification** | A large weight amplifies the importance of an input |
| **Signal Suppression** | A weight near 0 reduces the input's influence |
| **Sign Determination** | Negative weights can invert the effect of an input |
| **Learning Mechanism** | Weights are updated during training via backpropagation |
| **Feature Extraction** | Weights collectively allow the network to learn patterns |

### Key Insight
Initially, weights are set randomly. During training, the optimizer adjusts them to **minimize the loss function**, effectively teaching the network which features matter most.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def neuron_output(inputs, weights, bias):
    """Compute weighted sum (before activation)"""
    return np.dot(inputs, weights) + bias

inputs  = np.array([0.5, 0.8, 0.3])
weights = np.array([0.9, 0.1, 0.5])
bias    = 0.1

z = neuron_output(inputs, weights, bias)
print(f"Inputs:      {inputs}")
print(f"Weights:     {weights}")
print(f"Bias:        {bias}")
print(f"Weighted sum z = {z:.4f}")
print()

contributions = inputs * weights
print("Contribution of each input:")
for i, (inp, w, c) in enumerate(zip(inputs, weights, contributions)):
    print(f"  x{i+1}={inp} x w{i+1}={w} -> {c:.4f}")
print(f"  bias -> {bias}")
print(f"  Total z = {z:.4f}")

labels = ['x1*w1', 'x2*w2', 'x3*w3', 'bias']
values = list(contributions) + [bias]
colors = ['steelblue' if v >= 0 else 'tomato' for v in values]

plt.figure(figsize=(7, 4))
plt.bar(labels, values, color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Contribution of Each Weighted Input to Neuron Output')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

---
## Q2: What is an Activation Function?

An **activation function** is a non-linear mathematical function applied to the weighted sum of a neuron's inputs. It determines whether and how strongly a neuron activates.

### Why is it needed?
Without activation functions, a neural network regardless of its depth would behave as a single linear transformation and could not learn complex patterns.

### Common Activation Functions

| Function | Formula | Range | Use Case |
|----------|---------|-------|----------|
| **Sigmoid** | $\sigma(z) = \frac{1}{1+e^{-z}}$ | (0, 1) | Binary classification output |
| **Tanh** | $\tanh(z)$ | (-1, 1) | Hidden layers |
| **ReLU** | $f(z) = \max(0, z)$ | [0, inf) | Most hidden layers (default) |
| **Leaky ReLU** | $f(z) = \max(0.01z, z)$ | (-inf, inf) | Avoids dying ReLU problem |
| **Softmax** | $\frac{e^{z_i}}{\sum e^{z_j}}$ | (0, 1) | Multi-class output layer |

### Key Properties
- **Non-linearity**: Allows networks to model complex relationships
- **Differentiability**: Required for backpropagation
- **Monotonicity**: Helps ensure stable gradient flow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):     return 1 / (1 + np.exp(-z))
def tanh(z):        return np.tanh(z)
def relu(z):        return np.maximum(0, z)
def leaky_relu(z):  return np.where(z > 0, z, 0.01 * z)

z = np.linspace(-5, 5, 300)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle('Common Activation Functions', fontsize=14, fontweight='bold')

funcs  = [sigmoid, tanh, relu, leaky_relu]
names  = ['Sigmoid', 'Tanh', 'ReLU', 'Leaky ReLU']
colors = ['steelblue', 'darkorange', 'forestgreen', 'crimson']

for ax, f, name, color in zip(axes.flat, funcs, names, colors):
    ax.plot(z, f(z), color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('z')
    ax.set_ylabel('f(z)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Sigmoid output for z=0:", sigmoid(0))
print("ReLU output for z=-3: ", relu(-3))
print("ReLU output for z=3:  ", relu(3))

---
## Q3: Probability Distribution in ML Context

### Definition
A **probability distribution** is a mathematical function that describes the likelihood of different outcomes for a random variable. In ML, it models uncertainty over data, model parameters, and predictions.

### Key Types

| Type | Description | ML Usage |
|------|-------------|----------|
| **Bernoulli** | Binary outcome (0 or 1) | Binary classification |
| **Gaussian (Normal)** | Bell-shaped, continuous | Feature assumptions, weight initialization |
| **Categorical** | Multiple discrete outcomes | Multi-class classification |
| **Uniform** | Equal probability for all outcomes | Random initialization, dropout |
| **Binomial** | Number of successes in n trials | Aggregated binary outcomes |

### Why It Matters in ML
- **Loss functions** are derived from probability distributions (e.g., cross-entropy from Bernoulli)
- **Model outputs** are often probability distributions (e.g., softmax gives categorical distribution)
- **Bayesian ML** places distributions over model parameters
- **Generative models** (GANs, VAEs) involve sampling from distributions

### Important Concepts
- **PDF**: Probability Density Function for continuous variables
- **PMF**: Probability Mass Function for discrete variables
- **CDF**: Cumulative Distribution Function — probability that variable <= x

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Probability Distributions in ML', fontsize=13, fontweight='bold')

# 1. Gaussian Distribution
x = np.linspace(-4, 4, 300)
for mu, std, label in [(0, 1, 'mu=0, sigma=1'), (0, 0.5, 'mu=0, sigma=0.5'), (1, 1, 'mu=1, sigma=1')]:
    axes[0].plot(x, stats.norm.pdf(x, mu, std), label=label)
axes[0].set_title('Gaussian (Normal) Distribution')
axes[0].set_xlabel('x')
axes[0].set_ylabel('PDF')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# 2. Bernoulli Distribution
bar_width = 0.2
offsets = [-0.2, 0, 0.2]
ps = [0.3, 0.5, 0.7]
colors_b = ['steelblue', 'darkorange', 'forestgreen']
for offset, p, color in zip(offsets, ps, colors_b):
    axes[1].bar([0 + offset, 1 + offset], [1-p, p], width=bar_width,
                alpha=0.85, label=f'p={p}', color=color)
axes[1].set_title('Bernoulli Distribution')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['0 (Negative)', '1 (Positive)'])
axes[1].set_ylabel('Probability')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis='y')

# 3. Categorical (Softmax output)
classes = ['Cat', 'Dog', 'Bird', 'Fish']
probs   = [0.60, 0.25, 0.10, 0.05]
colors_c = ['steelblue', 'darkorange', 'forestgreen', 'crimson']
axes[2].bar(classes, probs, color=colors_c, edgecolor='black')
axes[2].set_title('Categorical Distribution\n(Softmax Output Example)')
axes[2].set_ylabel('Probability')
axes[2].set_ylim(0, 0.75)
for i, p in enumerate(probs):
    axes[2].text(i, p + 0.01, f'{p:.2f}', ha='center', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## Q4: Gradient in Optimization

### Definition
A **gradient** is a vector of partial derivatives of a function with respect to its parameters. In optimization, it points in the direction of **steepest increase** of the loss function.

### Mathematical Definition
For a loss function $L(w_1, w_2, ..., w_n)$, the gradient is:

$$\nabla L = \left(\frac{\partial L}{\partial w_1}, \frac{\partial L}{\partial w_2}, ..., \frac{\partial L}{\partial w_n}\right)$$

### Gradient Descent Update Rule
To minimize the loss, we move **opposite** to the gradient:

$$w \leftarrow w - \eta \cdot \nabla_w L$$

where $\eta$ (eta) is the **learning rate** — a hyperparameter controlling step size.

### Types of Gradient Descent

| Type | Batch Size | Pros | Cons |
|------|-----------|------|------|
| **Batch GD** | Full dataset | Stable convergence | Slow for large data |
| **Stochastic GD (SGD)** | 1 sample | Fast updates | Noisy, unstable |
| **Mini-batch GD** | 32–256 samples | Best of both worlds | Requires tuning |

### Role in Neural Networks
1. **Forward pass**: Compute predictions and loss
2. **Backward pass (Backpropagation)**: Compute gradients using the chain rule
3. **Weight update**: Adjust weights using computed gradients

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def loss(w):      return (w - 3) ** 2 + 2
def gradient(w):  return 2 * (w - 3)

w = -2.0
lr = 0.15
history = [w]

for _ in range(20):
    w = w - lr * gradient(w)
    history.append(w)

w_range = np.linspace(-3, 8, 300)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Gradient Descent Optimization', fontsize=13, fontweight='bold')

# Left: Loss landscape
axes[0].plot(w_range, loss(w_range), 'b-', linewidth=2, label='Loss L(w) = (w-3)^2 + 2')
axes[0].plot(history, [loss(h) for h in history], 'ro-', markersize=5, linewidth=1.5, label='GD path')
axes[0].scatter(history[0],  loss(history[0]),  s=150, color='green', zorder=5, label='Start')
axes[0].scatter(history[-1], loss(history[-1]), s=150, color='red',   zorder=5, label='End')
axes[0].set_xlabel('Weight w')
axes[0].set_ylabel('Loss L(w)')
axes[0].set_title('Gradient Descent on Loss Surface')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Loss over iterations
axes[1].plot(range(len(history)), [loss(h) for h in history],
             'darkorange', linewidth=2.5, marker='o', markersize=4)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Decreasing Over Iterations')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Starting weight: -2.0  ->  Final weight: {history[-1]:.4f}  (true minimum: 3.0)")
print(f"Starting loss:   {loss(-2.0):.4f} ->  Final loss:   {loss(history[-1]):.6f}")

---
## Q5: Logistic Regression Project with TensorFlow

### Project: Breast Cancer Classification

We build a **Logistic Regression** model using TensorFlow to classify breast cancer tumors as **Malignant** or **Benign** using the Wisconsin Breast Cancer Dataset.

**Logistic Regression** applies the sigmoid function:
$$\hat{y} = \sigma(Xw + b) = \frac{1}{1 + e^{-(Xw + b)}}$$

**Binary Cross-Entropy Loss:**
$$L = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i) \right]$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")

In [ ]:
# Step 1: Load and Explore the Dataset
data = load_breast_cancer()
X, y = data.data, data.target

df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y

print("Dataset Shape:", X.shape)
print("Features:     ", X.shape[1])
print("Samples:      ", X.shape[0])
print()
print("Class distribution:")
print(f"  Benign (0):    {(y == 0).sum()} samples")
print(f"  Malignant (1): {(y == 1).sum()} samples")
print()
print(df.describe().round(2))

In [ ]:
# Step 2: Preprocess the Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Convert to TensorFlow tensors
X_train_tensor = tf.constant(X_train, dtype=tf.float32)
X_test_tensor  = tf.constant(X_test,  dtype=tf.float32)
y_train_tensor = tf.constant(y_train, dtype=tf.float32)
y_test_tensor  = tf.constant(y_test,  dtype=tf.float32)

print(f"Training set size:  {X_train.shape[0]} samples")
print(f"Test set size:      {X_test.shape[0]} samples")
print(f"Feature dimensions: {X_train.shape[1]}")
print()
print(f"X_train_tensor: {X_train_tensor.shape}")
print(f"y_train_tensor: {y_train_tensor.shape}")

In [ ]:
# Step 3: Build the Logistic Regression Model
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(1, activation='sigmoid')
], name='Logistic_Regression')

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Step 4: Train the Model
history = model.fit(
    X_train_tensor, y_train_tensor,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    verbose=1
)

In [ ]:
# Step 5: Evaluate the Model
test_loss, test_acc = model.evaluate(X_test_tensor, y_test_tensor, verbose=0)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss:     {test_loss:.4f}")
print()

y_pred_proba = model.predict(X_test_tensor, verbose=0).flatten()
y_pred       = (y_pred_proba >= 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

In [ ]:
# Step 6: Visualize Results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Logistic Regression — Breast Cancer Classification Results', fontsize=13, fontweight='bold')

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Acc',  color='steelblue',  linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Acc',    color='darkorange', linewidth=2)
axes[0].set_title('Model Accuracy Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', color='steelblue',  linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='darkorange', linewidth=2)
axes[1].set_title('Model Loss Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
im = axes[2].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
axes[2].set_title('Confusion Matrix')
axes[2].set_xlabel('Predicted Label')
axes[2].set_ylabel('True Label')
axes[2].set_xticks([0, 1])
axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(['Benign', 'Malignant'])
axes[2].set_yticklabels(['Benign', 'Malignant'])
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black',
                     fontsize=18, fontweight='bold')
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Logistic Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Inspect Learned Weights (Tensors)
weights, bias = model.layers[0].get_weights()

print(f"Weight tensor shape: {weights.shape}")
print(f"Bias value:          {bias[0]:.4f}")
print()

weight_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Weight':  weights.flatten()
}).sort_values('Weight', key=abs, ascending=False)

print("Top 10 Most Influential Features:")
print(weight_df.head(10).to_string(index=False))

# Visualize
top10  = weight_df.head(10)
colors = ['steelblue' if w > 0 else 'tomato' for w in top10['Weight']]

plt.figure(figsize=(10, 5))
plt.barh(top10['Feature'], top10['Weight'], color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Weight Value')
plt.title('Top 10 Feature Weights — Logistic Regression')
plt.tight_layout()
plt.show()

---
## Summary

| Concept | Key Takeaway |
|--------|-------------|
| **Weights** | Learnable parameters that scale input signals; updated during training via backpropagation |
| **Activation Functions** | Introduce non-linearity so networks can learn complex patterns |
| **Probability Distributions** | Model uncertainty in data; underlie loss functions and model outputs |
| **Gradient** | Points in direction of steepest ascent; we move opposite to it to minimize loss |
| **Logistic Regression** | Single-neuron model with sigmoid activation for binary classification |

### Project Results
- **Dataset**: Wisconsin Breast Cancer (569 samples, 30 features)
- **Model**: Logistic Regression as a single Dense layer in TensorFlow
- **Optimizer**: Stochastic Gradient Descent (SGD), learning rate = 0.01
- **Loss**: Binary Cross-Entropy
- **Achieved**: ~95%+ test accuracy with high ROC-AUC score